# PT Monitor & Learning

Loads yesterday's race verdicts + dividends, calculates P&L, generates learnings via Claude, and updates the learning database.

**Cells:**
- 3a: Imports & config
- 3b: Load dividends + race verdict JSONs, join on raceId/saddle
- 3c: Calculate P&L → append to PT/betsMonitor.csv
- 3d: Generate learnings via Claude API
- 3e: Merge into PT/learnings_db.json


In [ ]:
# papermill parameters — injected by PT_WORKFLOW.ipynb when run automatically
ANTHROPIC_API_KEY = None


## 3a. Imports & Config

In [ ]:
# Standard library
import os, json, re, time
from datetime import datetime, timedelta, date

import numpy as np
import pandas as pd

!pip install -q anthropic
import anthropic


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
BASE  = '/content/drive/MyDrive/PT'
TODAY = date.today().strftime('%Y-%m-%d')
YESTERDAY = (date.today() - timedelta(days=1)).strftime('%Y-%m-%d')

# Override YESTERDAY if needed
# YESTERDAY = '2026-04-23'

if not ANTHROPIC_API_KEY:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', '')
# Or set directly:
# ANTHROPIC_API_KEY = 'sk-ant-...'

print(f'Analysing date: {YESTERDAY}')
print(f'Learnings DB: {BASE}/learnings_db.json')


## 3b. Load Dividends + Race Verdict JSONs

In [ ]:
# ── Load dividends ───────────────────────────────────────────────────────────
dividends = pd.read_parquet(f'{BASE}/dividends.parquet').drop_duplicates()
dividends['date'] = pd.to_datetime(dividends['date']).dt.strftime('%Y-%m-%d')
div_yday = dividends[dividends['date'] == YESTERDAY].copy()
print(f'Dividends for {YESTERDAY}: {len(div_yday)} rows')
div_yday.head(3)


In [ ]:
# ── Load race verdict JSONs for yesterday ────────────────────────────────────
import glob

_rv_dir = f'{BASE}/race_verdicts'
_json_files = glob.glob(f'{_rv_dir}/{YESTERDAY}_*.json')
print(f'Found {len(_json_files)} verdict JSON files for {YESTERDAY}')

all_race_verdicts = []  # flat list of race dicts
for _jf in _json_files:
    with open(_jf) as _f:
        _races = json.load(_f)
    all_race_verdicts.extend(_races)
    print(f'  {os.path.basename(_jf)}: {len(_races)} races')

print(f'Total races with verdicts: {len(all_race_verdicts)}')


In [ ]:
# ── Extract winners and placed horses from dividends ─────────────────────────
# winner: dividendType=='G' AND originBetType=='S'
# placed: dividendType=='P' AND originBetType=='S'
_win_div  = div_yday[(div_yday['dividendType'] == 'G') & (div_yday['originBetType'] == 'S')][['raceId', 'combination', 'dividend']]
_plc_div  = div_yday[(div_yday['dividendType'] == 'P') & (div_yday['originBetType'] == 'S')][['raceId', 'combination', 'dividend']]

_win_div.columns  = ['raceId', 'saddle', 'win_dividend']
_plc_div.columns  = ['raceId', 'saddle', 'plc_dividend']

# Normalise types
_win_div['raceId'] = _win_div['raceId'].astype(str)
_win_div['saddle'] = _win_div['saddle'].astype(str)
_plc_div['raceId'] = _plc_div['raceId'].astype(str)
_plc_div['saddle'] = _plc_div['saddle'].astype(str)

print(f'Winners found: {len(_win_div)}')
print(f'Placers found: {len(_plc_div)}')


## 3c. Calculate P&L

In [ ]:
# ── Build bets table from race verdicts ─────────────────────────────────────
rows = []
for race in all_race_verdicts:
    _rid      = str(race.get('raceId') or '')
    _meeting  = race.get('meeting', '')
    _race_name= race.get('race', '')
    _nap      = race.get('nap') or {}
    _ew       = race.get('each_way') or {}

    # Find saddle for NAP and EW horses
    _starters  = {s['name']: str(s.get('saddle') or '') for s in race.get('starters', [])}
    _nap_horse = _nap.get('horse', '')
    _ew_horse  = _ew.get('horse', '')
    _nap_saddle= _starters.get(_nap_horse, '')
    _ew_saddle = _starters.get(_ew_horse, '')

    rows.append({
        'datum':          YESTERDAY,
        'raceId':         _rid,
        'course':         _meeting,
        'race':           _race_name,
        'nap_horse':      _nap_horse,
        'nap_confidence': _nap.get('confidence'),
        'nap_saddle':     _nap_saddle,
        'ew_horse':       _ew_horse,
        'ew_confidence':  _ew.get('confidence'),
        'ew_saddle':      _ew_saddle,
    })

bets_df = pd.DataFrame(rows)
bets_df['raceId'] = bets_df['raceId'].astype(str)
print(f'Bets loaded: {len(bets_df)} races')
bets_df.head(3)


In [ ]:
# ── Join with dividends to determine win/place results ────────────────────────
bets_df = bets_df.merge(_win_div, left_on=['raceId', 'nap_saddle'],
                         right_on=['raceId', 'saddle'], how='left').rename(columns={'win_dividend': 'nap_win_div'})
bets_df = bets_df.drop(columns=['saddle'], errors='ignore')

bets_df = bets_df.merge(_plc_div, left_on=['raceId', 'nap_saddle'],
                         right_on=['raceId', 'saddle'], how='left').rename(columns={'plc_dividend': 'nap_plc_div'})
bets_df = bets_df.drop(columns=['saddle'], errors='ignore')

bets_df = bets_df.merge(_win_div, left_on=['raceId', 'ew_saddle'],
                         right_on=['raceId', 'saddle'], how='left').rename(columns={'win_dividend': 'ew_win_div'})
bets_df = bets_df.drop(columns=['saddle'], errors='ignore')

bets_df = bets_df.merge(_plc_div, left_on=['raceId', 'ew_saddle'],
                         right_on=['raceId', 'saddle'], how='left').rename(columns={'plc_dividend': 'ew_plc_div'})
bets_df = bets_df.drop(columns=['saddle'], errors='ignore')

# Result flags
bets_df['nap_result'] = bets_df['nap_win_div'].notna().map({True: 'WIN', False: 'LOSS'})
bets_df['ew_result']  = bets_df.apply(
    lambda r: 'WIN' if pd.notna(r['ew_win_div']) else ('PLACE' if pd.notna(r['ew_plc_div']) else 'LOSS'), axis=1
)

# P&L: NAP = 1 unit win; EW = 2 units (1 win + 1 place), -1 stake each leg
# Dividends are already the net return per unit (e.g. 5.50 = return 5.50 on 1 stake)
bets_df['nap_pnl'] = bets_df.apply(
    lambda r: (r['nap_win_div'] - 1) if pd.notna(r['nap_win_div']) else -1.0, axis=1
)
bets_df['ew_pnl'] = bets_df.apply(
    lambda r: (
        (r['ew_win_div'] - 1) + (r['ew_plc_div'] - 1) if pd.notna(r['ew_win_div'])
        else (r['ew_plc_div'] - 1) if pd.notna(r['ew_plc_div'])
        else -2.0
    ), axis=1
)

print(f'NAP results: {bets_df["nap_result"].value_counts().to_dict()}')
print(f'EW results:  {bets_df["ew_result"].value_counts().to_dict()}')
print(f'NAP P&L: {bets_df["nap_pnl"].sum():.2f} units')
print(f'EW P&L:  {bets_df["ew_pnl"].sum():.2f} units')
bets_df[['datum','course','nap_horse','nap_result','nap_pnl','ew_horse','ew_result','ew_pnl']]


In [ ]:
# ── Append to betsMonitor.csv ─────────────────────────────────────────────────
_monitor_path = f'{BASE}/betsMonitor.csv'
_cols = ['datum','raceId','course','race','nap_horse','nap_confidence','nap_result','nap_pnl',
          'ew_horse','ew_confidence','ew_result','ew_pnl']
_out = bets_df[[c for c in _cols if c in bets_df.columns]].copy()

if os.path.exists(_monitor_path):
    _existing = pd.read_csv(_monitor_path)
    # Remove any existing rows for YESTERDAY to avoid duplicates
    _existing = _existing[_existing['datum'] != YESTERDAY]
    _combined = pd.concat([_existing, _out], ignore_index=True)
else:
    _combined = _out

_combined.to_csv(_monitor_path, index=False)
print(f'✅ Saved betsMonitor.csv ({len(_combined)} total rows, {len(_out)} new for {YESTERDAY})')


## 3d. Generate Learnings via Claude

In [ ]:
# ── Build context: predictions + results ─────────────────────────────────────
_prediction_context = []
for race in all_race_verdicts:
    _rid  = str(race.get('raceId') or '')
    _nap  = race.get('nap') or {}
    _ew   = race.get('each_way') or {}
    _row  = bets_df[bets_df['raceId'] == _rid]
    if _row.empty:
        continue
    _r = _row.iloc[0]
    _prediction_context.append({
        'raceId':         _rid,
        'meeting':        race.get('meeting'),
        'race':           race.get('race'),
        'nap_horse':      _nap.get('horse'),
        'nap_confidence': _nap.get('confidence'),
        'nap_reason':     _nap.get('reason'),
        'nap_result':     _r.get('nap_result'),
        'nap_pnl':        round(float(_r.get('nap_pnl', 0)), 2),
        'ew_horse':       _ew.get('horse'),
        'ew_confidence':  _ew.get('confidence'),
        'ew_reason':      _ew.get('reason'),
        'ew_result':      _r.get('ew_result'),
        'ew_pnl':         round(float(_r.get('ew_pnl', 0)), 2),
    })

print(f'Sending {len(_prediction_context)} races to Claude for learnings...')


In [ ]:
# ── Claude API call for generalised learnings ─────────────────────────────────
_LEARNING_SYSTEM_PROMPT = '''You are an expert horse racing data analyst.

You receive a list of race predictions (NAP + Each Way) made by an AI tipster, along with
the actual results and P&L. Your task is to identify generalised, actionable learnings
that can improve future selections.

RULES:
  - Extract patterns, NOT race-specific observations. E.g.:
    GOOD: "High-confidence (>=8) NAPs with positive draw bias tend to win."
    BAD:  "Horse X won at Longchamp on 2026-04-23."
  - Each learning must be generalised and actionable.
  - Assign each learning an id (short snake_case string), direction (POSITIVE/NEGATIVE/NEUTRAL),
    category (trainer/jockey/going/draw/form/confidence/class/distance/sire/other), and
    a counter of 1 (newly observed).
  - Only emit a learning if you are confident it is supported by the data.
  - Aim for 3-10 learnings per session.

OUTPUT FORMAT — return ONLY valid JSON, no markdown:
[
  {
    "id": "short_snake_case_id",
    "learning": "Generalised actionable pattern.",
    "counter": 1,
    "direction": "POSITIVE",
    "category": "going"
  }
]
'''

_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
_resp = _client.messages.create(
    model='claude-sonnet-4-6',
    max_tokens=2048,
    system=_LEARNING_SYSTEM_PROMPT,
    messages=[{
        'role': 'user',
        'content': (
            f'Date: {YESTERDAY}\n'
            f'Predictions and results:\n'
            + json.dumps(_prediction_context, indent=2, default=str)
        )
    }],
)

_raw = _resp.content[0].text.strip()
_raw = re.sub(r'^```(?:json)?\s*', '', _raw)
_raw = re.sub(r'\s*```$', '', _raw)
_match = re.search(r'\[\s*\{[\s\S]*\}\s*\]', _raw)
new_learnings = []
if _match:
    try:
        new_learnings = json.loads(_match.group())
    except json.JSONDecodeError as _e:
        print(f'JSON parse error: {_e}')

print(f'✅ Generated {len(new_learnings)} new learnings')
for _l in new_learnings:
    print(f'  [{_l["direction"]}] {_l["id"]}: {_l["learning"]}')


## 3e. Merge Learnings into DB

In [ ]:
# ── Load existing learnings DB ────────────────────────────────────────────────
_db_path = f'{BASE}/learnings_db.json'
learnings_db = []
if os.path.exists(_db_path):
    with open(_db_path) as _f:
        learnings_db = json.load(_f)
print(f'Existing learnings: {len(learnings_db)}')

# ── Merge: same id → increment counter; new → add ────────────────────────────
_existing_ids = {l['id']: i for i, l in enumerate(learnings_db)}

for _nl in new_learnings:
    _nid = _nl.get('id', '')
    if _nid in _existing_ids:
        _idx = _existing_ids[_nid]
        learnings_db[_idx]['counter'] = learnings_db[_idx].get('counter', 1) + 1
        learnings_db[_idx]['last_updated'] = YESTERDAY
        print(f'  Updated: {_nid} (counter={learnings_db[_idx]["counter"]})')
    else:
        _nl['last_updated'] = YESTERDAY
        learnings_db.append(_nl)
        _existing_ids[_nid] = len(learnings_db) - 1
        print(f'  Added:   {_nid}')

# Sort by counter descending (most confirmed first)
learnings_db.sort(key=lambda x: x.get('counter', 0), reverse=True)

with open(_db_path, 'w', encoding='utf-8') as _f:
    json.dump(learnings_db, _f, ensure_ascii=False, indent=2)

print(f'\n✅ learnings_db.json saved: {len(learnings_db)} total learnings')
print(f'   Top 5 learnings by counter:')
for _l in learnings_db[:5]:
    print(f'   [{_l["counter"]}x] {_l["id"]}: {_l["learning"][:80]}')


## Summary

In [ ]:
print('\n✅ PT Monitor & Learning complete')
print(f'   Date analysed:    {YESTERDAY}')
print(f'   Races processed:  {len(all_race_verdicts)}')
print(f'   betsMonitor.csv:  {BASE}/betsMonitor.csv')
print(f'   learnings_db.json:{BASE}/learnings_db.json')
